# JSBSim C172 Empirical and do_trim Search v41

This notebook does not use PINN or MPC. It uses explicit for-loop sweeps over speed, elevator, and throttle to find an empirical level-flight starting condition for JSBSim C172. The best candidates are then re-run for a longer hold validation.

## 0. Install & Imports

In [ ]:
!pip install jsbsim -q
print('Install complete')


In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jsbsim

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)
    COLAB_DRIVE_MOUNTED = False

FPS_PER_KT = 1.687809857


## 1. Configuration

In [ ]:
EXPERIMENT_NAME = 'jsbsim_c172_empirical_and_dotrim_search_v41'
RUN_MODE = 'empirical_and_dotrim_grid'

DT = 0.02
INIT_ALT_FT = 3000.0
INIT_PITCH_DEG = 0.0

# Coarse grid first. Keep this small enough for Colab.
SPEED_GRID_KTS = np.arange(75.0, 116.0, 5.0)
ELEVATOR_GRID = np.round(np.arange(-0.25, 0.251, 0.05), 3)
THROTTLE_GRID = np.round(np.arange(0.45, 1.001, 0.05), 3)

COARSE_HOLD_SECONDS = 20.0
VALIDATION_HOLD_SECONDS = 60.0
TOP_K = 12

RUN_DOTRIM_GRID = True
DOTRIM_MODES = [0, 1, 2, 3, 4]
DOTRIM_SPEED_GRID_KTS = np.arange(70.0, 121.0, 5.0)
DOTRIM_THROTTLE_GRID = np.round(np.arange(0.45, 1.001, 0.05), 3)
DOTRIM_ELEVATOR_INIT_GRID = np.round(np.arange(-0.20, 0.201, 0.10), 3)
DOTRIM_HOLD_SECONDS = 30.0
DOTRIM_TOP_K = 12

# Optional local refinement around the best coarse result.
RUN_REFINEMENT = True
REFINE_SPEED_STEP_KTS = 2.0
REFINE_ELEVATOR_STEP = 0.015
REFINE_THROTTLE_STEP = 0.02
REFINE_RADIUS = 2
REFINE_HOLD_SECONDS = 30.0

TAIL_SECONDS = 6.0
TARGET_ALT_FT = INIT_ALT_FT
TARGET_SPEED_KTS = None  # None means use each case's initial speed as target.

WEIGHT_ALT_DRIFT = 1.0 / 20.0
WEIGHT_TAIL_VS = 1.0 / 1.0
WEIGHT_SPEED_DRIFT = 1.0 / 2.0
WEIGHT_Q_RMS = 1.0 / 0.5
WEIGHT_PITCH_ABS = 1.0 / 8.0
WEIGHT_ALPHA_ABS = 1.0 / 8.0

RESULT_ROOT = '/content/drive/MyDrive/Colab Result' if COLAB_DRIVE_MOUNTED else './Colab Result'
SAVE_DIR = os.path.join(RESULT_ROOT, 'PINN_MPC', EXPERIMENT_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print('Grid size:', len(SPEED_GRID_KTS) * len(ELEVATOR_GRID) * len(THROTTLE_GRID))
print('SAVE_DIR:', SAVE_DIR)


## 2. JSBSim Helpers

In [ ]:
def get_prop(fdm, name, default=0.0):
    try:
        return float(fdm[name])
    except Exception:
        return float(default)


def make_fdm(init_speed_kts, init_alt_ft=INIT_ALT_FT, init_pitch_deg=INIT_PITCH_DEG, init_elevator=0.0, init_throttle=None):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model('c172p')
    fdm['ic/h-sl-ft'] = float(init_alt_ft)
    fdm['ic/vt-kts'] = float(init_speed_kts)
    fdm['ic/theta-deg'] = float(init_pitch_deg)
    fdm['ic/psi-true-deg'] = 0.0
    fdm.run_ic()
    if init_throttle is None:
        init_throttle = INIT_THROTTLE
    fdm['propulsion/engine/set-running'] = 1
    fdm['fcs/elevator-cmd-norm'] = float(np.clip(init_elevator, -1.0, 1.0))
    fdm['fcs/throttle-cmd-norm'] = float(np.clip(init_throttle, 0.0, 1.0))
    return fdm


def read_state(fdm):
    speed_fps = get_prop(fdm, 'velocities/vt-fps')
    return {
        'alt_ft': get_prop(fdm, 'position/h-sl-ft'),
        'speed_fps': speed_fps,
        'speed_kts': speed_fps / FPS_PER_KT,
        'theta_deg': np.degrees(get_prop(fdm, 'attitude/theta-rad')),
        'q_deg_s': np.degrees(get_prop(fdm, 'velocities/q-rad_sec')),
        'alpha_deg': np.degrees(get_prop(fdm, 'aero/alpha-rad', 0.0)),
    }


def step_fixed(fdm, elevator, throttle):
    fdm['propulsion/engine/set-running'] = 1
    fdm['fcs/elevator-cmd-norm'] = float(np.clip(elevator, -1.0, 1.0))
    fdm['fcs/throttle-cmd-norm'] = float(np.clip(throttle, 0.0, 1.0))
    fdm.run()


def run_hold_case(init_speed_kts, elevator, throttle, seconds, label='case', store_log=False):
    fdm = make_fdm(init_speed_kts)
    steps = max(1, int(float(seconds) / DT))
    rows = [] if store_log else None
    alt0 = speed0 = theta0 = None
    tail_states = []
    for k in range(steps):
        s = read_state(fdm)
        if k == 0:
            alt0 = s['alt_ft']; speed0 = s['speed_kts']; theta0 = s['theta_deg']
        if store_log:
            row = dict(s)
            row.update({'time_s': k * DT, 'label': label, 'elevator': elevator, 'throttle': throttle, 'init_speed_kts': init_speed_kts})
            rows.append(row)
        if k >= steps - max(1, int(TAIL_SECONDS / DT)):
            tail_states.append(s)
        step_fixed(fdm, elevator, throttle)
    sf = read_state(fdm)
    tail_df = pd.DataFrame(tail_states)
    tail_vs = (tail_df['alt_ft'].iloc[-1] - tail_df['alt_ft'].iloc[0]) / max((len(tail_df)-1) * DT, DT) if len(tail_df) > 1 else np.nan
    q_rms = float(np.sqrt(np.mean(tail_df['q_deg_s'].values ** 2))) if len(tail_df) else np.nan
    target_speed = init_speed_kts if TARGET_SPEED_KTS is None else TARGET_SPEED_KTS
    alt_drift = float(sf['alt_ft'] - alt0)
    speed_drift = float(sf['speed_kts'] - target_speed)
    pitch_mean_abs = float(np.mean(np.abs(tail_df['theta_deg'].values))) if len(tail_df) else np.nan
    alpha_max_abs = float(np.max(np.abs(tail_df['alpha_deg'].values))) if len(tail_df) else np.nan
    score = (
        (WEIGHT_ALT_DRIFT * alt_drift) ** 2 +
        (WEIGHT_TAIL_VS * tail_vs) ** 2 +
        (WEIGHT_SPEED_DRIFT * speed_drift) ** 2 +
        (WEIGHT_Q_RMS * q_rms) ** 2 +
        (WEIGHT_PITCH_ABS * pitch_mean_abs) ** 2 +
        (WEIGHT_ALPHA_ABS * alpha_max_abs) ** 2
    )
    summary = {
        'label': label,
        'init_speed_kts': float(init_speed_kts),
        'elevator': float(elevator),
        'throttle': float(throttle),
        'seconds': float(seconds),
        'score': float(score),
        'alt0_ft': float(alt0),
        'alt_final_ft': float(sf['alt_ft']),
        'alt_drift_ft': alt_drift,
        'speed0_kts': float(speed0),
        'speed_final_kts': float(sf['speed_kts']),
        'speed_drift_kts': speed_drift,
        'theta0_deg': float(theta0),
        'theta_final_deg': float(sf['theta_deg']),
        'tail_vs_fps': float(tail_vs),
        'tail_q_rms_deg_s': q_rms,
        'tail_pitch_mean_abs_deg': pitch_mean_abs,
        'tail_alpha_max_abs_deg': alpha_max_abs,
    }
    return summary, pd.DataFrame(rows) if store_log else None


## 3. JSBSim do_trim Grid Sweep

This section changes the initial speed, throttle seed, elevator seed, and JSBSim trim mode, then calls `fdm.do_trim(mode)`. Successful and failed solver attempts are both logged. Every returned trim state is validated by fixed-control hold, because a solver return alone is not enough.

In [ ]:
def run_dotrim_seed_case(init_speed_kts, init_elevator, init_throttle, trim_mode, hold_seconds=DOTRIM_HOLD_SECONDS):
    fdm = make_fdm(init_speed_kts, init_elevator=init_elevator, init_throttle=init_throttle)
    trim_status = 'not_run'
    trim_exception = ''
    solver_returned = False
    try:
        result = fdm.do_trim(int(trim_mode))
        trim_status = f'returned:{result}'
        solver_returned = True
    except Exception as exc:
        trim_status = f'exception:{type(exc).__name__}:{exc}'
        trim_exception = str(exc)
    s = read_state(fdm)
    trim_elevator = float(get_prop(fdm, 'fcs/elevator-cmd-norm', init_elevator))
    trim_throttle = float(get_prop(fdm, 'fcs/throttle-cmd-norm', init_throttle))
    label = f'dotrim_m{trim_mode}_V{init_speed_kts:.0f}_e{init_elevator:.2f}_t{init_throttle:.2f}'
    summary, _ = run_hold_case(s['speed_kts'], trim_elevator, trim_throttle, hold_seconds, label=label, store_log=False)
    summary.update({
        'trim_method': 'jsbsim_do_trim_grid',
        'trim_mode': int(trim_mode),
        'solver_returned': bool(solver_returned),
        'trim_status': trim_status,
        'trim_exception': trim_exception,
        'seed_speed_kts': float(init_speed_kts),
        'seed_elevator': float(init_elevator),
        'seed_throttle': float(init_throttle),
        'trim_speed_kts': float(s['speed_kts']),
        'trim_pitch_deg': float(s['theta_deg']),
        'trim_elevator': trim_elevator,
        'trim_throttle': trim_throttle,
    })
    return summary


def run_dotrim_grid():
    rows = []
    total = len(DOTRIM_MODES) * len(DOTRIM_SPEED_GRID_KTS) * len(DOTRIM_ELEVATOR_INIT_GRID) * len(DOTRIM_THROTTLE_GRID)
    print('do_trim grid size:', total)
    t0 = time.time()
    idx = 0
    for mode in DOTRIM_MODES:
        for speed_kts in DOTRIM_SPEED_GRID_KTS:
            for init_elevator in DOTRIM_ELEVATOR_INIT_GRID:
                for init_throttle in DOTRIM_THROTTLE_GRID:
                    idx += 1
                    row = run_dotrim_seed_case(speed_kts, init_elevator, init_throttle, mode, DOTRIM_HOLD_SECONDS)
                    rows.append(row)
                    if idx % 100 == 0:
                        df_tmp = pd.DataFrame(rows).sort_values('score')
                        best = df_tmp.iloc[0]
                        n_return = int(pd.DataFrame(rows)['solver_returned'].sum())
                        print(f'do_trim {idx}/{total} elapsed={time.time()-t0:.1f}s returned={n_return} best score={best.score:.3f} mode={best.trim_mode} seedV={best.seed_speed_kts:.1f} trim_e={best.trim_elevator:.3f} trim_t={best.trim_throttle:.3f}')
    return pd.DataFrame(rows).sort_values('score').reset_index(drop=True)

if RUN_DOTRIM_GRID:
    dotrim_df = run_dotrim_grid()
else:
    dotrim_df = pd.DataFrame()

dotrim_path = os.path.join(SAVE_DIR, f'dotrim_grid_sweep_v41_{RUN_MODE}.csv')
dotrim_df.to_csv(dotrim_path, index=False)
print('Saved do_trim grid:', dotrim_path)
print('do_trim solver_returned counts:')
display(dotrim_df['solver_returned'].value_counts(dropna=False).to_frame('count') if len(dotrim_df) else dotrim_df)
print('Top do_trim candidates by hold validation score:')
display(dotrim_df.head(DOTRIM_TOP_K))


## 4. Coarse For-Loop Grid Search

In [ ]:
rows = []
t0 = time.time()
case_idx = 0
n_total = len(SPEED_GRID_KTS) * len(ELEVATOR_GRID) * len(THROTTLE_GRID)
for speed_kts in SPEED_GRID_KTS:
    for elevator in ELEVATOR_GRID:
        for throttle in THROTTLE_GRID:
            case_idx += 1
            summary, _ = run_hold_case(speed_kts, elevator, throttle, COARSE_HOLD_SECONDS, label='coarse')
            rows.append(summary)
            if case_idx % 100 == 0:
                best_so_far = min(rows, key=lambda r: r['score'])
                print(f'{case_idx}/{n_total} elapsed={time.time()-t0:.1f}s best score={best_so_far["score"]:.3f} V={best_so_far["init_speed_kts"]:.1f} elev={best_so_far["elevator"]:.3f} thr={best_so_far["throttle"]:.3f}')

coarse_df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
coarse_path = os.path.join(SAVE_DIR, f'empirical_trim_coarse_grid_v40_{RUN_MODE}.csv')
coarse_df.to_csv(coarse_path, index=False)
print('Saved coarse grid:', coarse_path)
print('Top coarse candidates:')
display(coarse_df.head(TOP_K))


## 5. Local Refinement Around Best Candidate

In [ ]:
if RUN_REFINEMENT:
    best = coarse_df.iloc[0]
    speed_values = np.round(best['init_speed_kts'] + REFINE_SPEED_STEP_KTS * np.arange(-REFINE_RADIUS, REFINE_RADIUS + 1), 3)
    elevator_values = np.round(best['elevator'] + REFINE_ELEVATOR_STEP * np.arange(-REFINE_RADIUS, REFINE_RADIUS + 1), 4)
    throttle_values = np.round(best['throttle'] + REFINE_THROTTLE_STEP * np.arange(-REFINE_RADIUS, REFINE_RADIUS + 1), 4)
    speed_values = speed_values[(speed_values >= 60.0) & (speed_values <= 130.0)]
    elevator_values = np.clip(elevator_values, -0.5, 0.5)
    throttle_values = np.clip(throttle_values, 0.0, 1.0)
    refine_rows = []
    n_total = len(speed_values) * len(elevator_values) * len(throttle_values)
    print('Refinement grid size:', n_total)
    case_idx = 0
    for speed_kts in speed_values:
        for elevator in elevator_values:
            for throttle in throttle_values:
                case_idx += 1
                summary, _ = run_hold_case(speed_kts, elevator, throttle, REFINE_HOLD_SECONDS, label='refine')
                refine_rows.append(summary)
    refine_df = pd.DataFrame(refine_rows).sort_values('score').reset_index(drop=True)
else:
    refine_df = coarse_df.head(TOP_K).copy()

refine_path = os.path.join(SAVE_DIR, f'empirical_trim_refine_grid_v40_{RUN_MODE}.csv')
refine_df.to_csv(refine_path, index=False)
print('Saved refine grid:', refine_path)
print('Top refined candidates:')
display(refine_df.head(TOP_K))


## 6. Long Hold Validation for Top Candidates

In [ ]:
validation_logs = []
validation_rows = []
for rank, row in refine_df.head(TOP_K).iterrows():
    label = f'valid_rank_{rank+1}_V{row.init_speed_kts:.1f}_e{row.elevator:.3f}_t{row.throttle:.3f}'
    summary, log_df = run_hold_case(row.init_speed_kts, row.elevator, row.throttle, VALIDATION_HOLD_SECONDS, label=label, store_log=True)
    summary['rank'] = rank + 1
    validation_rows.append(summary)
    validation_logs.append(log_df)

validation_df = pd.DataFrame(validation_rows).sort_values('score').reset_index(drop=True)
validation_log_df = pd.concat(validation_logs, ignore_index=True)
validation_path = os.path.join(SAVE_DIR, f'empirical_trim_validation_summary_v40_{RUN_MODE}.csv')
validation_log_path = os.path.join(SAVE_DIR, f'empirical_trim_validation_logs_v40_{RUN_MODE}.csv')
validation_df.to_csv(validation_path, index=False)
validation_log_df.to_csv(validation_log_path, index=False)
print('Saved validation summary:', validation_path)
print('Saved validation logs:', validation_log_path)
print('Best validation candidates:')
display(validation_df.head(TOP_K))

best_trim = validation_df.iloc[0].to_dict()
best_json_path = os.path.join(SAVE_DIR, f'empirical_trim_best_candidate_v40_{RUN_MODE}.json')
with open(best_json_path, 'w') as f:
    json.dump(best_trim, f, indent=2)
print('Best candidate json:', best_json_path)
print(json.dumps(best_trim, indent=2))


## 6.1 do_trim vs Empirical Best Comparison

In [ ]:
comparison_frames = []
if len(dotrim_df):
    a = dotrim_df.head(DOTRIM_TOP_K).copy()
    a['source'] = 'jsbsim_do_trim'
    comparison_frames.append(a)
b = validation_df.head(TOP_K).copy()
b['source'] = 'empirical_fixed_control'
comparison_frames.append(b)
combined_best_df = pd.concat(comparison_frames, ignore_index=True).sort_values('score').reset_index(drop=True)
combined_path = os.path.join(SAVE_DIR, f'empirical_vs_dotrim_best_comparison_v41_{RUN_MODE}.csv')
combined_best_df.to_csv(combined_path, index=False)
print('Saved combined comparison:', combined_path)
display(combined_best_df.head(2 * TOP_K))


## 7. Plots

In [ ]:
def plot_validation(validation_log_df, top_n=5):
    labels = validation_df.head(top_n)['label'].tolist()
    fig, axes = plt.subplots(5, 1, figsize=(12, 13), sharex=True)
    for label in labels:
        df = validation_log_df[validation_log_df['label'].eq(label)]
        axes[0].plot(df['time_s'], df['alt_ft'], label=label)
        axes[1].plot(df['time_s'], df['speed_kts'], label=label)
        axes[2].plot(df['time_s'], df['theta_deg'], label=label)
        axes[3].plot(df['time_s'], df['q_deg_s'], label=label)
        axes[4].plot(df['time_s'], df['elevator'], label=f'{label} elev')
        axes[4].plot(df['time_s'], df['throttle'], ls='--', label=f'{label} thr')
    axes[0].axhline(INIT_ALT_FT, color='k', ls='--', lw=0.8)
    axes[0].set_ylabel('Altitude (ft)')
    axes[1].set_ylabel('Speed (kt)')
    axes[2].set_ylabel('Pitch (deg)')
    axes[3].set_ylabel('q (deg/s)')
    axes[4].set_ylabel('Controls')
    axes[4].set_xlabel('Time (s)')
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', fontsize=8)
    fig.suptitle('Empirical trim validation: top fixed-control candidates')
    plt.tight_layout()
    return fig

fig = plot_validation(validation_log_df, top_n=min(5, len(validation_df)))
plot_path = os.path.join(SAVE_DIR, f'empirical_trim_validation_top_candidates_v40_{RUN_MODE}.png')
fig.savefig(plot_path, dpi=160)
print('Saved plot:', plot_path)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc = axes[0].scatter(coarse_df['elevator'], coarse_df['throttle'], c=np.clip(coarse_df['score'], 0, coarse_df['score'].quantile(0.9)), s=16, cmap='viridis')
axes[0].set_xlabel('Elevator')
axes[0].set_ylabel('Throttle')
axes[0].set_title('Coarse score projection')
plt.colorbar(sc, ax=axes[0], label='score clipped')
axes[1].scatter(coarse_df['init_speed_kts'], coarse_df['score'], s=10, alpha=0.4)
axes[1].set_xlabel('Initial speed (kt)')
axes[1].set_ylabel('Score')
axes[1].set_title('Score vs speed')
axes[1].grid(True, alpha=0.25)
axes[2].bar(np.arange(min(10, len(validation_df))), validation_df['score'].head(10))
axes[2].set_xlabel('Validation rank')
axes[2].set_ylabel('Score')
axes[2].set_title('Top validation scores')
axes[2].grid(True, axis='y', alpha=0.25)
plt.tight_layout()
score_plot_path = os.path.join(SAVE_DIR, f'empirical_trim_score_diagnostics_v40_{RUN_MODE}.png')
fig.savefig(score_plot_path, dpi=160)
print('Saved score plot:', score_plot_path)
plt.show()


if len(dotrim_df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(dotrim_df['solver_returned'].astype(str).value_counts().index, dotrim_df['solver_returned'].astype(str).value_counts().values)
    axes[0].set_title('do_trim solver returned')
    axes[0].set_ylabel('count')
    axes[1].scatter(dotrim_df['seed_speed_kts'], dotrim_df['score'], s=10, alpha=0.35)
    axes[1].set_xlabel('Seed speed (kt)')
    axes[1].set_ylabel('hold-validation score')
    axes[1].set_title('do_trim score vs seed speed')
    colors = dotrim_df['solver_returned'].map({True: 'tab:green', False: 'tab:red'}).fillna('tab:gray')
    axes[2].scatter(dotrim_df['trim_elevator'], dotrim_df['trim_throttle'], c=colors, s=10, alpha=0.45)
    axes[2].set_xlabel('trim/readback elevator')
    axes[2].set_ylabel('trim/readback throttle')
    axes[2].set_title('do_trim readback controls')
    for ax in axes:
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    dotrim_plot_path = os.path.join(SAVE_DIR, f'dotrim_grid_diagnostics_v41_{RUN_MODE}.png')
    fig.savefig(dotrim_plot_path, dpi=160)
    print('Saved do_trim diagnostics:', dotrim_plot_path)
    plt.show()


## 8. How To Use The Result

Use the best validation row as the empirical trim condition for later PINN-MPC notebooks. Recommended mapping: `INIT_SPEED_KTS = init_speed_kts`, `TRIM_ELEVATOR_CMD = elevator`, `TRIM_THROTTLE_CMD = throttle`, and pitch reference offset from the early hold pitch if needed. If the best 60 s validation still has large altitude drift or tail vertical speed, expand the grid or reduce the maneuver speed target before comparing controllers.